In [1]:
import os
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer, logging, RobertaModel
import torch

from dataset import CL_Dataset, CustomSubset
from evaluation import compute_cdcr_metrics
from model import SiameseNetwork, CustomContrastiveLoss

logging.set_verbosity_error()

In [3]:
## Parameters
#Data

train_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/train']
val_conll_paths = ['../CDCR/data/ECBplus-prep/output_data/val']
mention_type = "event"
context_method = "hsu"
train_dataset_name = "ecbplus_train"
val_dataset_name = "ecbplus_val"
language_model = 'roberta-base'
special_tokens = ('[E]', '[/E]')

#Model
gpu_num = [0]
margin=0.4

#Optimizer and learning
learning_rate = 2e-7
weight_decay = 1e-2
#num_epochs = 100
num_epochs = 3
train_batch_size = 4
val_batch_size = 32

In [4]:
#Initialize model and tokenizer
roberta = RobertaModel.from_pretrained(pretrained_model_name_or_path= language_model) 
tokenizer = RobertaTokenizer.from_pretrained(language_model)
tokenizer.add_tokens(special_tokens)

2

In [5]:
#Initialization
#Load data

cl_cdcr_train = CL_Dataset(name=train_dataset_name,data_paths=train_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="train") 

cl_cdcr_val = CL_Dataset(name=val_dataset_name,data_paths=val_conll_paths,context_method=context_method,
                           mention_type=mention_type,tokenizer=tokenizer, special_mention_tokens=special_tokens,mode="inference") 

#Use subset for testing:
cl_cdcr_val = CustomSubset(cl_cdcr_val,[0,1,2,3],mode="inference")
cl_cdcr_train = CustomSubset(cl_cdcr_train,[0,1,2,3,4,5,6,7,8],mode="train")
#Keep in mind ECB topic repeated in train and topic. topic 34 I think

train_dataloader= DataLoader(cl_cdcr_train, batch_size=train_batch_size, shuffle=True)

val_dataloader= DataLoader(cl_cdcr_val, batch_size=val_batch_size, shuffle=False)

In [6]:
len(train_dataloader)

3

In [13]:
# Initialize model
device_ids = gpu_num
device = torch.device("cuda:{}".format(device_ids[0]))
#Temporary use of cpu
device = torch.device("cpu")

siamese_model_single = SiameseNetwork(model=roberta,tokenizer=tokenizer).to(device)
model=siamese_model_single
model_device = next(model.parameters()).device
print(model_device)
# To paralelize
#siamese_model = torch.nn.DataParallel(siamese_model_single, device_ids=device_ids)
#model = siamese_model_single


#

cpu


In [8]:
# Initialize optimizer
#Parameters
# Define the learning rate and weight decay

# Define optimizer!
# AdamW optimizer
# Epochs: 100 Events 50 Entities
# Batch size 16
# m 0.4 0.7
# Clistering Threshold
# Learning rate  2e-7 2e-7


optimizer_params = [
    {'params': model.language_model.parameters(), 'lr': learning_rate},
    {'params': model.mlp_cls_embed.parameters(), 'lr': learning_rate},
    {'params': model.mlp_mention_embed.parameters(), 'lr': learning_rate},
]

contrastive_loss = CustomContrastiveLoss(margin=margin)
optimizer = torch.optim.AdamW(optimizer_params)

In [11]:
#Training loop

best_conll_f1 = 0.0
model_path = os.path.join(os.getcwd(), "models/")
if not os.path.exists(model_path):
    os.makedirs(model_path)

for epoch in range(num_epochs):
    out_dir = os.path.join(model_path,f"experiment",f"roberta_{train_batch_size}",f"checkpoint_{epoch}")
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    accumulate_loss = 0
    model.train()

    #Train dataloader
    for array_mention1,array_mention2, labels in train_dataloader:

        #¿ Can I move arrays to device?
        array_mention1.to(device)
        array_mention2.to(device)
        
        optimizer.zero_grad()  # Zero the gradients
        output_mentions1 = model(array_mention1)
        output_mentions2 = model(array_mention2)
        
        loss = contrastive_loss(output_mentions1, output_mentions2, labels)

        loss.backward()  # Backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()  # Optimization step

        torch.cuda.empty_cache()
        accumulate_loss += loss.item()
        print("train dataloader iteration")
    
    print(f'epoch {epoch+1}/{num_epochs}, Train accumulate loss: {accumulate_loss}')

    #Validation
    model.eval()
    #Store vector representations by topic
    df_metrics = compute_cdcr_metrics(cl_cdcr_dataset=cl_cdcr_val,dataloader=val_dataloader,model=model,metrics="all")
    #Store best model, but first correct the CoNLL score
    conll_f1 = df_metrics.loc["conll","F1"]
    print(f'epoch {epoch+1}/{num_epochs}, eval conll: {conll_f1}')
    if conll_f1 > best_conll_f1:
        best_conll_f1 = conll_f1
        torch.save(model.state_dict(), out_dir+"/siamese_model.pth")
        print(f'epoch {epoch+1}/{num_epochs}, model saved')



<class 'torch.Tensor'>
cpu


KeyboardInterrupt: 

In [16]:
# Load the model

loaded_siamese_model = SiameseNetwork(model=roberta,tokenizer=tokenizer)

# Load the saved parameters
loaded_siamese_model.load_state_dict(torch.load(out_dir+'/siamese_model.pth'))

<All keys matched successfully>